# Phase 12: Strong one-hot baselines: MPRA-LegNet and MPRAnn

**Question.** Tang et al. (*Genome Biology* 2025) found that representations from pretrained DNA language models, probed without updating the language-model weights, did not clearly beat one-hot CNNs on lentiMPRA activity, and that fine-tuning raised performance to a level comparable with their strong ResNet. Phase 9B/9C compared fine-tuned DNABERT-2 only with spaced-mer ridge and a small 6-epoch CNN. This phase extends that comparison to this project's chromosome and locus splits and its independent external cohort. It adds the two published MPRA-specialist architectures trained on this same assay family, under the **unchanged** Phase 9B (chromosome) and Phase 9C (locus) frozen splits, and scores them on the frozen Phase 10 CODA cohort.

**Models (fixed before any result is seen).**
1. `legnet`: MPRA-LegNet architecture from `autosome-ru/human_legnet` (MIT, Penzar et al.), repository defaults: stem 64/11, EffBlock kernel 9, blocks [80, 96, 112, 128], pools [2,2,2,2], resize 4; AdamW + OneCycle (max_lr 0.01, wd 0.1), batch 1024, 20 epochs.
2. `mprann`: MPRAnn "standard" architecture from `visze/sequence_cnn_models` (MIT, Agarwal et al.), ported from Keras to PyTorch; Adam 1e-3, batch 128, up to 100 epochs, patience 10.

Both use three declared seeds (20260930, 20260931, 20260932), train-only target scaling, random strand per epoch during training, forward/RC mean at inference, and **validation-Pearson** epoch selection, exactly as Phase 9B/9C. Test chromosomes, test loci and CODA are scored only after every validation choice is complete.

**Deviations from the original training recipes (declared):** inputs are the 200 bp genomic sequences without assay adapters, so LegNet's adapter shift augmentation and reverse-channel input are not used; MPRAnn early stopping monitors validation Pearson instead of validation loss, to match the Phase 9 selection rule. LegNet keeps its original gradient clipping (norm 1); MPRAnn's original has none.

**Statistical scope:** bootstrap intervals are conditional on these trained models and these partitions; they do not capture all training variability, so individual seeds are reported alongside. The external primary interval resamples sequences; a region-grouped bootstrap (natural regions within 1 kb resampled together) is reported as a sensitivity check.

**Primary prespecified contrast:** per cell line, Phase 9B chromosome-test Pearson of the DNABERT-2 three-seed mean minus the LegNet three-seed mean, paired locus-group bootstrap (1,000 draws). Secondary: the same against MPRAnn; Phase 9C locus test; Phase 10 DHS-natural Spearman. All are reported whatever their direction.

### Inputs (attach as Kaggle Inputs, or place in a local folder)
1. `9_enhancer_phase9a_full_activity_dataset.zip`
2. `9B_enhancer_phase9b_regression_results.zip`
3. `9C_enhancer_phase9c_regression_results.zip`
4. `10_enhancer_phase10_external_transfer_results.zip`

No checkpoints or Internet are needed. **Kaggle:** GPU on, Internet off is fine. **Local PC:** set `PHASE12_INPUT` to the folder holding the four ZIPs (default `./phase12_inputs`). Interrupted runs resume: completed jobs are verified and reused.

Download `enhancer_phase12_strong_baselines_results.zip` (small) and `enhancer_phase12_complete_with_weights.zip`.

In [1]:
# 0. Imports, frozen protocol and input discovery.
import os, sys, io, json, zipfile, hashlib, gc, random, shutil, math, time
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

SMOKE = os.environ.get('PHASE12_SMOKE', '0') == '1'   # tiny CPU check only; never a result
if Path('/kaggle/input').exists():
    ROOT = Path('/kaggle/input'); WORK_BASE = Path('/kaggle/working')
else:
    ROOT = Path(os.environ.get('PHASE12_INPUT', './phase12_inputs')).resolve()
    WORK_BASE = Path(os.environ.get('PHASE12_OUTPUT', './phase12_outputs')).resolve()
assert ROOT.exists(), f'Input folder not found: {ROOT}'
WORK = WORK_BASE / ('enhancer_phase12_SMOKE_work' if SMOKE else 'enhancer_phase12_work')
CACHE = WORK / 'cache'; WEIGHTS = WORK / 'checkpoints'; PRED = WORK / 'predictions'
for p in (WORK, CACHE, WEIGHTS, PRED): p.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    'phase9a_audit_and_provenance.json': '39118ed7b2dc50631f87dcb0c470249c649609d3dd7ab8c885b10064c66deff2',
    'phase9_full_activity_all_cells.csv': '4a359a715f7a3f1fbcf0d044cc08446137df584c9ab4a43a2de18637bb341a76',
    'phase9_frozen_splits.csv': '1ecc2d271817edd3000890be718a3f3a9bcc6b63a71550dfa718dbed84d0ff61',
    'phase9b_predictions_validation_test.csv': '8c506af49eb8b3baa82b0d8eebcb5fcdddf3c1e2a2b62bb4b280a37c48e183e3',
    'phase9c_predictions_validation_test.csv': '72a54ea366b057d825808c7e8662460b092a12ed99a8739e206b3ecd4a13561e',
    'phase10_external_frozen_set.csv': '00d3521b5f087df56e97b792e7b961ffcb9a80e346fdb4c8360dc0182dc273f8',
    'phase10_external_predictions.csv': 'a97db76c8784961bc12fcd35cf7b7d4f716d9c22d0bfcb8aaca29d334834c22f',
}
PARENT_LOCKS = {'9a': '5e438773a3f9c74cf006d183bcfc30daabd3d85bdbfefd47acfa48dc023e450a',
                '9b': '9a5c05aca44b1007301965ad5d7c7877629865071ad6fae88b2692d3a9cd199d',
                '9c': 'dd2b4c43281df1782900f5b11265b80d1539c635ee261fe82d07f70a8705f610',
                '10': '0de96f349bf5bc6342b45e449d1833ef397ea467a00f3e6b1d55f7371e9a7fa8'}
CONFIG = {
    'name': 'phase12_strong_onehot_baselines_v1',
    'parent_locks': PARENT_LOCKS, 'input_sha256': EXPECTED,
    'protocols': {'9b': 'split_9b', '9c': 'split_9a'},
    'cells': ['HepG2', 'K562'], 'seeds': [20260930, 20260931, 20260932],
    'legnet': {'source': 'github.com/autosome-ru/human_legnet@a20dbc7d0b08daae3af1aebcfb8a71d9f198ab5d model.py (MIT)',
               'in_ch': 4, 'stem_ch': 64, 'stem_ks': 11, 'ef_ks': 9, 'ef_block_sizes': [80, 96, 112, 128],
               'pool_sizes': [2, 2, 2, 2], 'resize_factor': 4, 'optimizer': 'AdamW lr=max_lr/25 + OneCycleLR pct_start 0.3',
               'max_lr': 0.01, 'weight_decay': 0.1, 'batch': 1024, 'epochs': 20, 'patience': None,
               'gradient_clip_norm': 1.0, 'clip_source': 'core.py pl.Trainer(gradient_clip_val=1)'},
    'mprann': {'source': 'github.com/visze/sequence_cnn_models@7638fce137db0445123efe8d7e2c35e248fafc5f lib/model.py standard() (MIT), Keras-to-PyTorch port',
               'optimizer': 'Adam', 'lr': 0.001, 'batch': 128, 'epochs': 100, 'patience': 10, 'gradient_clip_norm': None,
               'port_check': 'eval-mode outputs equal original Keras standard() with copied weights (max abs diff 2.7e-7)'},
    'selection': 'highest validation Pearson; ties keep earliest epoch',
    'rc_policy': 'random strand per record per epoch in training; mean of forward and RC at inference',
    'target_scaling': 'z-score with training mean and SD only',
    'primary_contrast': '9B chromosome test Pearson: dnabert2_three_seed_mean minus legnet_three_seed_mean, per cell',
    'secondary_contrasts': ['same vs mprann_three_seed_mean', '9C locus test', 'Phase 10 DHS_natural Spearman'],
    'bootstrap_repeats': 1000, 'bootstrap_seed': 20261012,
    'internal_resampling_unit': 'test locus_group_id', 'external_resampling_unit': 'unique external DNA sequence',
    'external_sensitivity_unit': 'DHS_natural hg38 intervals merged when on the same chromosome and within 1,000 bp',
    'interval_scope': 'bootstrap intervals are conditional on the fitted models and frozen partitions; seed variation is reported separately',
}
def sha(b): return hashlib.sha256(b).hexdigest()
def fsha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(8 * 1024 * 1024), b''): h.update(b)
    return h.hexdigest()
LOCK = sha(json.dumps(CONFIG, sort_keys=True, separators=(',', ':')).encode())
print('Phase 12 protocol:', LOCK, '| SMOKE' if SMOKE else '')

def _zip_hits(raw, name, where, depth=0):
    out = []
    try:
        with zipfile.ZipFile(io.BytesIO(raw) if isinstance(raw, bytes) else raw) as z:
            for m in z.namelist():
                if m == name or m.endswith('/' + name): out.append((z.read(m), where + '!' + m))
                elif m.lower().endswith('.zip') and depth < 1:
                    out += _zip_hits(z.read(m), name, where + '!' + m, depth + 1)
    except (zipfile.BadZipFile, OSError): pass
    return out
def locked_member(name):
    found = [(p.read_bytes(), str(p)) for p in ROOT.rglob(name) if p.is_file()]
    for p in ROOT.rglob('*.zip'):
        if p.is_file(): found += _zip_hits(p, name, str(p))
    good = [(b, w) for b, w in found if sha(b) == EXPECTED[name]]
    if not good: raise RuntimeError(f'Missing or wrong-version {name}. Seen: {[(w, sha(b)[:12]) for b, w in found]}')
    print('Verified', name, 'from', good[0][1]); return good[0][0]
RAW = {n: locked_member(n) for n in EXPECTED}
audit = json.loads(RAW['phase9a_audit_and_provenance.json']); assert audit['protocol_hash'] == PARENT_LOCKS['9a']
d = pd.read_csv(io.BytesIO(RAW['phase9_full_activity_all_cells.csv']),
                dtype={'record_id': str, 'sequence': str, 'locus_group_id': str, 'split_group_id': str, 'chrom': str})
locked = pd.read_csv(io.BytesIO(RAW['phase9_frozen_splits.csv']), dtype=str, keep_default_na=False)
assert len(d) == 76833 and d.record_id.is_unique and d.record_id.tolist() == locked.record_id.tolist()
for col in ('cell_line', 'locus_group_id', 'split_group_id', 'split_9a', 'split_9b'):
    assert d[col].astype(str).tolist() == locked[col].tolist(), col
assert d.sequence.str.fullmatch('[ACGT]{200}').all() and np.isfinite(d.mean_log2_activity).all()
assert d.groupby('locus_group_id').split_9b.nunique().max() == 1
assert d.groupby('split_group_id').split_9a.nunique().max() == 1
external = pd.read_csv(io.BytesIO(RAW['phase10_external_frozen_set.csv']), dtype={'record_id': str, 'sequence': str, 'origin': str})
assert len(external) == 29330 and external.record_id.is_unique and external.sequence.str.fullmatch('[ACGT]{200}').all()
PRIOR = {tag: pd.read_csv(io.BytesIO(RAW[f'phase{tag}_predictions_validation_test.csv']), dtype={'record_id': str})
         for tag in ('9b', '9c')}
EXT_PRIOR = pd.read_csv(io.BytesIO(RAW['phase10_external_predictions.csv']), dtype={'record_id': str, 'protocol': str})
print(d.groupby(['cell_line', 'split_9b']).size().to_dict())
print('External cohort:', external.origin.value_counts().to_dict())

# Import finished jobs from a previous run's saved Output, if attached.
for folder in (CACHE, WEIGHTS, PRED):
    prior = [p for p in ROOT.rglob('*') if p.is_file() and p.parent.name == folder.name and p.parent.parent.name == WORK.name]
    for p in prior:
        target = folder / p.name
        if target.exists(): assert fsha(target) == fsha(p), f'Conflicting previous {p.name}'
        else: shutil.copy2(p, target)

Phase 12 protocol: 1506ebc3e38db096dfa16fcfab1b0e29e8a7cefaa8f8255a678bdce012bbffcf 
Verified phase9a_audit_and_provenance.json from D:\Aditta Thesis\For Shafin\Shafin's Contribution\phase12_inputs\9_enhancer_phase9a_full_activity_dataset.zip!phase9a_audit_and_provenance.json
Verified phase9_full_activity_all_cells.csv from D:\Aditta Thesis\For Shafin\Shafin's Contribution\phase12_inputs\9_enhancer_phase9a_full_activity_dataset.zip!phase9_full_activity_all_cells.csv
Verified phase9_frozen_splits.csv from D:\Aditta Thesis\For Shafin\Shafin's Contribution\phase12_inputs\9_enhancer_phase9a_full_activity_dataset.zip!phase9_frozen_splits.csv
Verified phase9b_predictions_validation_test.csv from D:\Aditta Thesis\For Shafin\Shafin's Contribution\phase12_inputs\9B_enhancer_phase9b_regression_results.zip!phase9b_predictions_validation_test.csv
Verified phase9c_predictions_validation_test.csv from D:\Aditta Thesis\For Shafin\Shafin's Contribution\phase12_inputs\9C_enhancer_phase9c_regression_res

In [2]:
# 1. Shared helpers: frozen partitions, metrics, atomic job caches.
MODELS = ['legnet', 'mprann']; SEEDS = CONFIG['seeds']; CELLS = CONFIG['cells']; PROTOCOLS = ('9b', '9c')
COMP = str.maketrans('ACGT', 'TGCA')
def rc(s): return s.translate(COMP)[::-1]
def partitions(tag, cell):
    col = CONFIG['protocols'][tag]; sub = d[d.cell_line.eq(cell)]
    parts = [sub[sub[col].eq(s)].reset_index(drop=True) for s in ('train', 'validation', 'test')]
    if SMOKE: parts = [x.sample(n=min(len(x), k), random_state=0).sort_index().reset_index(drop=True) for x, k in zip(parts, (800, 200, 200))]
    return tuple(parts)
EXT = external.head(300).reset_index(drop=True) if SMOKE else external
def correlation(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float)
    if np.std(y) == 0 or np.std(p) == 0: return float('nan')   # undefined for constant input; never -1
    return float(np.corrcoef(y, p)[0, 1])
def measure(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float); assert len(y) == len(p) > 3 and np.isfinite(p).all()
    return {'n': int(len(y)), 'pearson': correlation(y, p), 'spearman': float(spearmanr(y, p).statistic),
            'r2': float(r2_score(y, p)), 'rmse': float(np.sqrt(mean_squared_error(y, p))), 'mae': float(mean_absolute_error(y, p))}
def write_csv(frame, path):
    tmp = path.with_suffix('.part'); frame.to_csv(tmp, index=False, float_format='%.12g'); os.replace(tmp, path)
def write_json(obj, path):
    tmp = path.with_suffix('.part'); tmp.write_text(json.dumps(obj, indent=2, sort_keys=True) + '\n'); os.replace(tmp, path)
def job(tag, cell, model, seed): return f'{tag}_{cell}_{model}_seed_{seed}'
def save_scores(name, kind, ids, pred, extra=None):
    pred = np.asarray(pred, float); assert len(pred) == len(ids) and np.isfinite(pred).all()
    p = PRED / f'{name}_{kind}.csv'; write_csv(pd.DataFrame({'record_id': ids, 'prediction': pred}), p)
    write_json({'lock': LOCK, 'job': name, 'kind': kind, 'sha256': fsha(p), **(extra or {})}, CACHE / f'{name}_{kind}.json')
def load_scores(name, kind, ids):
    p = PRED / f'{name}_{kind}.csv'; m = CACHE / f'{name}_{kind}.json'
    if not (p.exists() and m.exists()): return None
    meta = json.loads(m.read_text())
    if meta['lock'] != LOCK: return None
    assert meta['sha256'] == fsha(p), f'Corrupted cache {p.name}'
    x = pd.read_csv(p, dtype={'record_id': str})
    assert x.record_id.tolist() == list(ids) and np.isfinite(x.prediction).all()
    return x.prediction.to_numpy(), meta

In [3]:
# 2. Architectures.
import torch
from torch import nn
import torch.nn.functional as F
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if not SMOKE: assert DEVICE.type == 'cuda', 'Select a GPU accelerator (or run locally on a CUDA GPU).'
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
try: torch.use_deterministic_algorithms(True, warn_only=True)
except Exception: pass
print('Device:', DEVICE, torch.__version__)

# ---- MPRA-LegNet. Copied from autosome-ru/human_legnet model.py (commit a20dbc7).
# MIT License, Copyright 2024 Dmitry Penzar. Permission is hereby granted, free of charge, to any person
# obtaining a copy of this software ... The above copyright notice and this permission notice shall be
# included in all copies or substantial portions of the Software. (Full text: repository LICENSE.)
def initialize_weights(m):
    if isinstance(m, nn.Conv1d):
        n = m.kernel_size[0] * m.out_channels
        m.weight.data.normal_(0, math.sqrt(2 / n))
        if m.bias is not None: nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, nn.BatchNorm1d):
        nn.init.constant_(m.weight.data, 1); nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, nn.Linear):
        m.weight.data.normal_(0, 0.001)
        if m.bias is not None: nn.init.constant_(m.bias.data, 0)
class SELayer(nn.Module):
    def __init__(self, inp, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(inp, int(inp // reduction)), nn.SiLU(), nn.Linear(int(inp // reduction), inp), nn.Sigmoid())
    def forward(self, x):
        b, c, _, = x.size(); y = x.view(b, c, -1).mean(dim=2); y = self.fc(y).view(b, c, 1); return x * y
class EffBlock(nn.Module):
    def __init__(self, in_ch, ks, resize_factor, activation, out_ch=None, se_reduction=None):
        super().__init__()
        self.in_ch = in_ch; self.out_ch = self.in_ch if out_ch is None else out_ch
        self.resize_factor = resize_factor; self.se_reduction = resize_factor if se_reduction is None else se_reduction
        self.ks = ks; self.inner_dim = self.in_ch * self.resize_factor
        self.block = nn.Sequential(
            nn.Conv1d(self.in_ch, self.inner_dim, kernel_size=1, padding='same', bias=False), nn.BatchNorm1d(self.inner_dim), activation(),
            nn.Conv1d(self.inner_dim, self.inner_dim, kernel_size=ks, groups=self.inner_dim, padding='same', bias=False),
            nn.BatchNorm1d(self.inner_dim), activation(), SELayer(self.inner_dim, reduction=self.se_reduction),
            nn.Conv1d(self.inner_dim, self.in_ch, kernel_size=1, padding='same', bias=False), nn.BatchNorm1d(self.in_ch), activation())
    def forward(self, x): return self.block(x)
class LocalBlock(nn.Module):
    def __init__(self, in_ch, ks, activation, out_ch=None):
        super().__init__()
        self.in_ch = in_ch; self.out_ch = self.in_ch if out_ch is None else out_ch; self.ks = ks
        self.block = nn.Sequential(nn.Conv1d(self.in_ch, self.out_ch, kernel_size=self.ks, padding='same', bias=False),
                                   nn.BatchNorm1d(self.out_ch), activation())
    def forward(self, x): return self.block(x)
class ResidualConcat(nn.Module):
    def __init__(self, fn): super().__init__(); self.fn = fn
    def forward(self, x, **kwargs): return torch.concat([self.fn(x, **kwargs), x], dim=1)
class MapperBlock(nn.Module):
    def __init__(self, in_features, out_features, activation=nn.SiLU):
        super().__init__()
        self.block = nn.Sequential(nn.BatchNorm1d(in_features), nn.Conv1d(in_features, out_features, kernel_size=1))
    def forward(self, x): return self.block(x)
class LegNet(nn.Module):
    def __init__(self, in_ch, stem_ch, stem_ks, ef_ks, ef_block_sizes, pool_sizes, resize_factor, activation=nn.SiLU):
        super().__init__()
        assert len(pool_sizes) == len(ef_block_sizes)
        self.in_ch = in_ch
        self.stem = LocalBlock(in_ch=in_ch, out_ch=stem_ch, ks=stem_ks, activation=activation)
        blocks = []; in_ch = stem_ch; out_ch = stem_ch
        for pool_sz, out_ch in zip(pool_sizes, ef_block_sizes):
            blocks.append(nn.Sequential(
                ResidualConcat(EffBlock(in_ch=in_ch, out_ch=in_ch, ks=ef_ks, resize_factor=resize_factor, activation=activation)),
                LocalBlock(in_ch=in_ch * 2, out_ch=out_ch, ks=ef_ks, activation=activation),
                nn.MaxPool1d(pool_sz) if pool_sz != 1 else nn.Identity()))
            in_ch = out_ch
        self.main = nn.Sequential(*blocks)
        self.mapper = MapperBlock(in_features=out_ch, out_features=out_ch * 2)
        self.head = nn.Sequential(nn.Linear(out_ch * 2, out_ch * 2), nn.BatchNorm1d(out_ch * 2), activation(), nn.Linear(out_ch * 2, 1))
    def forward(self, x):
        x = self.stem(x); x = self.main(x); x = self.mapper(x)
        x = F.adaptive_avg_pool1d(x, 1).squeeze(-1); return self.head(x).squeeze(-1)
# ---- end of copied LegNet code.

class MPRAnn(nn.Module):
    """PyTorch port of `standard()` in visze/sequence_cnn_models lib/model.py (MIT, Max Schubach / BIH).
    Keras Conv1D uses 'valid' padding; activation precedes BatchNormalization; softmax is over channels;
    Keras BN defaults (momentum 0.99, eps 1e-3) and glorot_uniform/zero initialisation are reproduced."""
    def __init__(self, length=200):
        super().__init__()
        bn = lambda c: nn.BatchNorm1d(c, eps=1e-3, momentum=0.01)
        self.c1, self.b1 = nn.Conv1d(4, 250, 7), bn(250)
        self.c2, self.b2 = nn.Conv1d(250, 250, 8), bn(250)
        self.c3, self.b3 = nn.Conv1d(250, 250, 3), bn(250)
        self.c4, self.b4 = nn.Conv1d(250, 100, 2), bn(100)
        self.drop1, self.drop2, self.drop3 = nn.Dropout(0.1), nn.Dropout(0.1), nn.Dropout(0.3)
        L = ((length - 6 - 7) // 2) - 2 - 1
        self.d1, self.d2, self.out = nn.Linear(100 * L, 300), nn.Linear(300, 200), nn.Linear(200, 1)
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        x = self.b1(F.relu(self.c1(x)))
        x = self.b2(torch.softmax(self.c2(x), dim=1)); x = self.drop1(F.max_pool1d(x, 2))
        x = self.b3(torch.softmax(self.c3(x), dim=1))
        x = self.b4(torch.softmax(self.c4(x), dim=1)); x = self.drop2(x)
        x = x.transpose(1, 2).flatten(1)   # Keras Flatten order (length, channels)
        x = self.drop3(torch.sigmoid(self.d1(x))); x = torch.sigmoid(self.d2(x))
        return self.out(x).squeeze(-1)

def build(model):
    if model == 'legnet':
        c = CONFIG['legnet']
        net = LegNet(in_ch=c['in_ch'], stem_ch=c['stem_ch'], stem_ks=c['stem_ks'], ef_ks=c['ef_ks'],
                     ef_block_sizes=c['ef_block_sizes'], pool_sizes=c['pool_sizes'], resize_factor=c['resize_factor'])
        net.apply(initialize_weights); return net
    return MPRAnn()
for m in MODELS:
    n = sum(p.numel() for p in build(m).parameters()); print(m, 'parameters', f'{n:,}')

Device: cuda:0 2.11.0+cu128
legnet parameters 1,322,465
mprann parameters 3,507,751


In [4]:
# 3. Training with validation-only epoch selection. Every finished job is cached and reusable.
LUT = np.zeros(256, dtype=np.int64)
for b, i in zip(b'ACGT', range(4)): LUT[b] = i
EYE = np.eye(4, dtype=np.float32)
def onehot(seqs):
    codes = np.stack([LUT[np.frombuffer(s.encode(), dtype=np.uint8)] for s in seqs])
    return torch.from_numpy(EYE[codes].transpose(0, 2, 1).copy())
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
AMP = DEVICE.type == 'cuda'
def predict(net, seqs, mean, scale, batch=1024):
    net.eval(); out = []
    with torch.inference_mode():
        for i in range(0, len(seqs), batch):
            chunk = seqs[i:i + batch]
            x = onehot(chunk).to(DEVICE); xr = onehot([rc(s) for s in chunk]).to(DEVICE)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                p = net(x).float(); q = net(xr).float()
            out.append(((p + q) / 2 * scale + mean).cpu().numpy())
    return np.concatenate(out).astype(float)
def train_job(tag, cell, model, seed):
    name = job(tag, cell, model, seed); tr, va, te = partitions(tag, cell)
    ckpt = WEIGHTS / f'{name}_best.pt'
    cached = load_scores(name, 'validation', va.record_id)
    if cached is not None and ckpt.exists() and cached[1].get('checkpoint_sha256') == fsha(ckpt):
        print('REUSED', name); return cached[1]['choice']
    c = CONFIG[model]; set_seed(seed)
    y = tr.mean_log2_activity.to_numpy(np.float32); mean = float(y.mean()); scale = float(y.std()); assert scale > 0
    yz = torch.from_numpy((y - mean) / scale); seqs = tr.sequence.tolist()
    net = build(model).to(DEVICE)
    epochs = 1 if SMOKE else c['epochs']; bs = min(c['batch'], len(seqs))
    steps = len(seqs) // bs
    if model == 'legnet':
        opt = torch.optim.AdamW(net.parameters(), lr=c['max_lr'] / 25, weight_decay=c['weight_decay'])
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=c['max_lr'], three_phase=False,
                                                    total_steps=epochs * steps, pct_start=0.3, cycle_momentum=False)
    else:
        opt = torch.optim.Adam(net.parameters(), lr=c['lr'], eps=1e-7); sched = None
    scaler = torch.amp.GradScaler('cuda', enabled=AMP); gen = np.random.default_rng(seed)
    best = (-float('inf'), None, None); log = []; stale = 0; t0 = time.time()
    for epoch in range(1, epochs + 1):
        net.train(); order = gen.permutation(len(seqs)); flip = gen.random(len(seqs)) < 0.5; losses = []
        for s in range(steps):   # drop_last: BatchNorm in the LegNet head needs batches > 1
            idx = order[s * bs:(s + 1) * bs]
            x = onehot([rc(seqs[i]) if flip[i] else seqs[i] for i in idx]).to(DEVICE); t = yz[idx].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                loss = F.mse_loss(net(x).float(), t)
            assert torch.isfinite(loss), (name, epoch, s)
            scaler.scale(loss).backward()
            if c['gradient_clip_norm']:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(net.parameters(), c['gradient_clip_norm'])
            scaler.step(opt); scaler.update()
            if sched is not None: sched.step()
            losses.append(float(loss.detach()))
        pv = predict(net, va.sequence.tolist(), mean, scale); r = correlation(va.mean_log2_activity, pv)
        log.append({'epoch': epoch, 'train_loss': float(np.mean(losses)), 'validation_pearson': r, 'pred_sd': float(np.std(pv))})
        print(f'{name} epoch {epoch} loss {np.mean(losses):.4f} val r {r:.4f} sd {np.std(pv):.3f}', flush=True)
        if np.isfinite(r) and r > best[0]:   # a constant (collapsed) epoch can never be selected
            best = (r, epoch, pv); stale = 0
            tmp = ckpt.with_suffix('.part')
            torch.save({'lock': LOCK, 'job': name, 'epoch': epoch, 'target_mean': mean, 'target_scale': scale,
                        'state_dict': {k: v.detach().cpu() for k, v in net.state_dict().items()}}, tmp)
            os.replace(tmp, ckpt)
        else:
            stale += 1
            if c['patience'] and stale >= c['patience']: break
    if best[1] is None:   # no epoch gave a non-constant validation prediction: keep the last epoch, flag it, report it
        best = (float('nan'), epoch, pv)
        torch.save({'lock': LOCK, 'job': name, 'epoch': epoch, 'target_mean': mean, 'target_scale': scale,
                    'state_dict': {k: v.detach().cpu() for k, v in net.state_dict().items()}}, ckpt)
    choice = {'protocol': tag, 'cell_line': cell, 'model': model, 'seed': seed, 'selected_epoch': best[1], 'collapsed': not np.isfinite(best[0]),
              'validation_pearson': best[0], 'epochs_run': len(log), 'minutes': round((time.time() - t0) / 60, 2)}
    write_csv(pd.DataFrame(log), CACHE / f'{name}_training_log.csv')
    save_scores(name, 'validation', va.record_id, best[2], {'choice': choice, 'checkpoint_sha256': fsha(ckpt)})
    del net, opt; gc.collect(); torch.cuda.empty_cache()
    return choice
CHOICES = [train_job(tag, cell, model, seed) for tag in PROTOCOLS for cell in CELLS for model in MODELS for seed in SEEDS]
print(pd.DataFrame(CHOICES).to_string(index=False))

9b_HepG2_legnet_seed_20260930 epoch 1 loss 0.9477 val r 0.1762 sd 0.003
9b_HepG2_legnet_seed_20260930 epoch 2 loss 0.8304 val r 0.2028 sd 0.107
9b_HepG2_legnet_seed_20260930 epoch 3 loss 0.7470 val r 0.4320 sd 0.196
9b_HepG2_legnet_seed_20260930 epoch 4 loss 0.7221 val r 0.4191 sd 1.412
9b_HepG2_legnet_seed_20260930 epoch 5 loss 0.6716 val r 0.4669 sd 0.711
9b_HepG2_legnet_seed_20260930 epoch 6 loss 0.6693 val r 0.4445 sd 0.785
9b_HepG2_legnet_seed_20260930 epoch 7 loss 0.6222 val r 0.4708 sd 0.866
9b_HepG2_legnet_seed_20260930 epoch 8 loss 0.5515 val r 0.5369 sd 0.455
9b_HepG2_legnet_seed_20260930 epoch 9 loss 0.4968 val r 0.5709 sd 0.535
9b_HepG2_legnet_seed_20260930 epoch 10 loss 0.4666 val r 0.4935 sd 0.732
9b_HepG2_legnet_seed_20260930 epoch 11 loss 0.4025 val r 0.5694 sd 0.629
9b_HepG2_legnet_seed_20260930 epoch 12 loss 0.3627 val r 0.6099 sd 0.587
9b_HepG2_legnet_seed_20260930 epoch 13 loss 0.2932 val r 0.5907 sd 0.596
9b_HepG2_legnet_seed_20260930 epoch 14 loss 0.2354 val r 0.6

In [5]:
# 4. Locked inference: internal test partitions and the frozen Phase 10 cohort, only after ALL selections.
assert len(CHOICES) == len(PROTOCOLS) * len(CELLS) * len(MODELS) * len(SEEDS)
def load_net(name):
    ck = torch.load(WEIGHTS / f'{name}_best.pt', map_location='cpu', weights_only=True)
    assert ck['lock'] == LOCK and ck['job'] == name
    model = 'legnet' if '_legnet_' in name else 'mprann'
    net = build(model).to(DEVICE); net.load_state_dict(ck['state_dict'], strict=True)
    return net, ck['target_mean'], ck['target_scale']
for tag in PROTOCOLS:
    for cell in CELLS:
        tr, va, te = partitions(tag, cell)
        for model in MODELS:
            for seed in SEEDS:
                name = job(tag, cell, model, seed)
                need_test = load_scores(name, 'test', te.record_id) is None
                need_ext = load_scores(name, 'external', EXT.record_id) is None
                if not (need_test or need_ext): continue
                net, mean, scale = load_net(name)
                if need_test: save_scores(name, 'test', te.record_id, predict(net, te.sequence.tolist(), mean, scale))
                if need_ext: save_scores(name, 'external', EXT.record_id, predict(net, EXT.sequence.tolist(), mean, scale))
                del net; torch.cuda.empty_cache()
        print('Locked scores ready:', tag, cell, flush=True)

Locked scores ready: 9b HepG2
Locked scores ready: 9b K562
Locked scores ready: 9c HepG2
Locked scores ready: 9c K562


In [6]:
# 5. Internal metrics, joined with the existing Phase 9B/9C arms on identical records, and paired locus bootstrap.
rows = []
for tag in PROTOCOLS:
    for cell in CELLS:
        tr, va, te = partitions(tag, cell)
        for split, frame in (('validation', va), ('test', te)):
            vecs = {}
            for model in MODELS:
                for seed in SEEDS:
                    name = job(tag, cell, model, seed)
                    vecs[f'{model}_seed_{seed}'] = load_scores(name, split, frame.record_id)[0]
                vecs[f'{model}_three_seed_mean'] = np.mean([vecs[f'{model}_seed_{s}'] for s in SEEDS], axis=0)
            for arm, v in vecs.items():
                rows.append(pd.DataFrame({'protocol': tag, 'record_id': frame.record_id, 'cell_line': cell, 'chrom': frame.chrom,
                    'locus_group_id': frame.locus_group_id, 'split': split, 'model': arm,
                    'target': frame.mean_log2_activity, 'prediction': v}))
new_pred = pd.concat(rows, ignore_index=True)
prior = pd.concat([PRIOR[t].assign(protocol=t) for t in PROTOCOLS], ignore_index=True)
keys = new_pred[['protocol', 'split', 'record_id']].drop_duplicates()
prior = prior.merge(keys, on=['protocol', 'split', 'record_id'], how='inner', validate='many_to_one')
for (tag, split), part in prior.groupby(['protocol', 'split']):   # every prior arm must cover every new record
    need = len(keys[(keys.protocol == tag) & (keys.split == split)])
    assert part.groupby('model').record_id.nunique().eq(need).all(), (tag, split)
for (tag, split), part in new_pred.groupby(['protocol', 'split']):
    ref = prior[(prior.protocol == tag) & (prior.split == split) & (prior.model == 'spaced6_4_ridge')].set_index('record_id')
    sub = part[part.model == part.model.iloc[0]].set_index('record_id')
    assert ref.loc[sub.index].target.round(6).tolist() == sub.target.round(6).tolist(), f'Target mismatch {tag} {split}'
allpred = pd.concat([prior, new_pred], ignore_index=True)
metrics = pd.DataFrame([{'protocol': t, 'cell_line': c, 'split': s, 'model': m, **measure(p.target, p.prediction)}
                        for (t, c, s, m), p in allpred.groupby(['protocol', 'cell_line', 'split', 'model'])])
show = ['gc_cpg_ridge', 'spaced6_4_ridge', 'cnn_three_seed_mean', 'mprann_three_seed_mean', 'legnet_three_seed_mean', 'dnabert2_three_seed_mean']
print(metrics[metrics.split.eq('test') & metrics.model.isin(show)].sort_values(['protocol', 'cell_line', 'pearson'], ascending=[True, True, False])
      [['protocol', 'cell_line', 'model', 'n', 'pearson', 'spearman', 'r2', 'rmse']].to_string(index=False))
print('\nIndividual seeds (test):')
print(metrics[metrics.split.eq('test') & metrics.model.str.contains('_seed_')].pivot_table(
      index=['protocol', 'cell_line'], columns='model', values='pearson').round(4).T.to_string())

rng = np.random.default_rng(CONFIG['bootstrap_seed']); boot = []
CONTRASTS = [('dnabert2_three_seed_mean', 'legnet_three_seed_mean', 'primary' ), ('dnabert2_three_seed_mean', 'mprann_three_seed_mean', 'secondary'),
             ('legnet_three_seed_mean', 'spaced6_4_ridge', 'secondary'), ('mprann_three_seed_mean', 'spaced6_4_ridge', 'secondary'),
             ('legnet_three_seed_mean', 'cnn_three_seed_mean', 'secondary')]
for tag in PROTOCOLS:
    for cell in CELLS:
        sub = allpred[(allpred.protocol == tag) & (allpred.cell_line == cell) & (allpred.split == 'test')]
        wide = sub.pivot(index='record_id', columns='model', values='prediction')
        truth = sub[sub.model == 'spaced6_4_ridge'].set_index('record_id').loc[wide.index]
        y = truth.target.to_numpy(); groups = pd.Series(np.arange(len(y))).groupby(truth.locus_group_id.to_numpy()).apply(lambda v: v.to_numpy()).tolist()
        draws = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]) for _ in range(CONFIG['bootstrap_repeats'])]
        for a, b, role in CONTRASTS:
            pa, pb = wide[a].to_numpy(), wide[b].to_numpy()
            diffs = np.array([correlation(y[ix], pa[ix]) - correlation(y[ix], pb[ix]) for ix in draws])
            boot.append({'protocol': tag, 'cell_line': cell, 'contrast': f'{a} minus {b}',
                         'role': 'primary' if (role == 'primary' and tag == '9b') else 'secondary',
                         'pearson_difference': correlation(y, pa) - correlation(y, pb),
                         'lower_95': float(np.nanquantile(diffs, .025)), 'upper_95': float(np.nanquantile(diffs, .975)),
                         'bootstrap_repeats': len(diffs), 'undefined_draws': int(np.isnan(diffs).sum()),
                         'resampling_unit': CONFIG['internal_resampling_unit']})
internal_boot = pd.DataFrame(boot); print(internal_boot.to_string(index=False))

protocol cell_line                    model     n  pearson  spearman       r2     rmse
      9b     HepG2   legnet_three_seed_mean  9048 0.662449  0.665500 0.425844 0.537961
      9b     HepG2   mprann_three_seed_mean  9048 0.655661  0.659497 0.418503 0.541388
      9b     HepG2 dnabert2_three_seed_mean  9048 0.595329  0.594245 0.354144 0.570563
      9b     HepG2          spaced6_4_ridge  9048 0.526486  0.539646 0.277069 0.603648
      9b     HepG2      cnn_three_seed_mean  9048 0.506324  0.520422 0.237564 0.619922
      9b     HepG2             gc_cpg_ridge  9048 0.158237  0.180502 0.024983 0.701038
      9b      K562   legnet_three_seed_mean 13145 0.708002  0.638889 0.494527 0.305432
      9b      K562   mprann_three_seed_mean 13145 0.681506  0.605215 0.461942 0.315123
      9b      K562 dnabert2_three_seed_mean 13145 0.642160  0.573208 0.390134 0.335492
      9b      K562          spaced6_4_ridge 13145 0.535265  0.492002 0.285712 0.363080
      9b      K562      cnn_three_seed_mean

In [7]:
# 6. Phase 10 external cohort: Spearman by origin and paired bootstrap on DHS_natural, against the frozen Phase 10 scores.
ORIGINS = ['DHS_natural', 'Malinois_natural', 'AdaLead', 'FastSeqProp', 'Simulated_Annealing']
ext_rows = []; scores = {}
for tag in PROTOCOLS:
    for cell in CELLS:
        for model in MODELS:
            vs = [load_scores(job(tag, cell, model, s), 'external', EXT.record_id)[0] for s in SEEDS]
            for s, v in zip(SEEDS, vs): scores[tag, cell, f'{model}_seed_{s}'] = v
            scores[tag, cell, f'{model}_three_seed_mean'] = np.mean(vs, axis=0)
        old = EXT_PRIOR[(EXT_PRIOR.protocol == tag) & (EXT_PRIOR.cell_line == cell)]
        for arm, part in old.groupby('model'):
            part = part.set_index('record_id').loc[EXT.record_id]; scores[tag, cell, arm] = part.prediction.to_numpy()
for (tag, cell, arm), v in scores.items():
    truth = EXT[f'{cell}_l2fc'].to_numpy(float)
    for origin in ORIGINS:
        ix = np.flatnonzero(EXT.origin.eq(origin).to_numpy())
        if len(ix) < 4: continue
        ext_rows.append({'protocol': tag, 'cell_line': cell, 'origin': origin, 'primary': origin == 'DHS_natural', 'model': arm,
                         'n': len(ix), 'spearman': float(spearmanr(truth[ix], v[ix]).statistic), 'pearson': correlation(truth[ix], v[ix])})
ext_metrics = pd.DataFrame(ext_rows)
print(ext_metrics[ext_metrics.primary & ext_metrics.model.isin(show)].sort_values(['cell_line', 'protocol', 'spearman'], ascending=[True, True, False]).to_string(index=False))
ext_boot = []; ix = np.flatnonzero(EXT.origin.eq('DHS_natural').to_numpy())
# Sensitivity unit: nearby natural regions (same chromosome, within 1 kb) are resampled together.
dhs = EXT.iloc[ix][['external_chrom', 'external_start', 'external_end']].reset_index(drop=True)
assert dhs.external_chrom.str.startswith('chr').all() and (dhs.external_start >= 0).all()
region = np.empty(len(dhs), dtype=np.int64); rid = -1
for chrom, g in dhs.sort_values(['external_chrom', 'external_start']).groupby('external_chrom', sort=True):
    end = -10**12
    for i, st, en in zip(g.index, g.external_start, g.external_end):
        if st > end + 1000: rid += 1; end = en
        else: end = max(end, en)
        region[i] = rid
region_members = pd.Series(np.arange(len(region))).groupby(region).apply(lambda v: v.to_numpy()).tolist()
print('DHS_natural sequences', len(ix), 'grouped into', len(region_members), 'regions; largest region', max(map(len, region_members)))
for tag in PROTOCOLS:
    for cell in CELLS:
        y = EXT[f'{cell}_l2fc'].to_numpy(float)[ix]
        draws = [rng.integers(0, len(ix), len(ix)) for _ in range(CONFIG['bootstrap_repeats'])]
        region_draws = [np.concatenate([region_members[i] for i in rng.integers(0, len(region_members), len(region_members))])
                        for _ in range(CONFIG['bootstrap_repeats'])]
        for a, b, role in CONTRASTS:
            pa, pb = scores[tag, cell, a][ix], scores[tag, cell, b][ix]
            for unit, unit_draws in (('sequence', draws), ('region', region_draws)):
                diffs = np.array([spearmanr(y[k], pa[k]).statistic - spearmanr(y[k], pb[k]).statistic for k in unit_draws])
                ext_boot.append({'protocol': tag, 'cell_line': cell, 'contrast': f'{a} minus {b}', 'n': len(ix),
                                 'analysis': 'primary' if unit == 'sequence' else 'sensitivity',
                                 'spearman_difference': float(spearmanr(y, pa).statistic - spearmanr(y, pb).statistic),
                                 'lower_95': float(np.nanquantile(diffs, .025)), 'upper_95': float(np.nanquantile(diffs, .975)),
                                 'bootstrap_repeats': len(diffs), 'undefined_draws': int(np.isnan(diffs).sum()),
                                 'resampling_unit': CONFIG['external_resampling_unit'] if unit == 'sequence' else CONFIG['external_sensitivity_unit']})
external_boot = pd.DataFrame(ext_boot); print(external_boot.to_string(index=False))

protocol cell_line      origin  primary                    model    n  spearman  pearson
      9b     HepG2 DHS_natural     True   mprann_three_seed_mean 8399  0.534166 0.560167
      9b     HepG2 DHS_natural     True   legnet_three_seed_mean 8399  0.524558 0.551147
      9b     HepG2 DHS_natural     True dnabert2_three_seed_mean 8399  0.457040 0.480335
      9b     HepG2 DHS_natural     True          spaced6_4_ridge 8399  0.417550 0.425110
      9b     HepG2 DHS_natural     True      cnn_three_seed_mean 8399  0.321506 0.348865
      9b     HepG2 DHS_natural     True             gc_cpg_ridge 8399  0.101162 0.101483
      9c     HepG2 DHS_natural     True   mprann_three_seed_mean 8399  0.526426 0.555394
      9c     HepG2 DHS_natural     True   legnet_three_seed_mean 8399  0.522311 0.555608
      9c     HepG2 DHS_natural     True dnabert2_three_seed_mean 8399  0.466125 0.488104
      9c     HepG2 DHS_natural     True          spaced6_4_ridge 8399  0.420205 0.426938
      9c     HepG2 DH

In [8]:
# 7. Package results and weights. Smoke runs are labelled and must never be reported.
ART = WORK / 'results'; ART.mkdir(exist_ok=True); files = {}
def save(name, frame): p = ART / name; write_csv(frame, p); files[name] = p
save('phase12_internal_predictions_new_arms.csv', new_pred)
save('phase12_internal_metrics_all_arms.csv', metrics)
save('phase12_internal_paired_bootstrap.csv', internal_boot)
save('phase12_external_metrics_all_arms.csv', ext_metrics)
save('phase12_external_paired_bootstrap.csv', external_boot)
save('phase12_validation_choices.csv', pd.DataFrame(CHOICES))
logs = pd.concat([pd.read_csv(CACHE / f'{job(t, c, m, s)}_training_log.csv').assign(job=job(t, c, m, s))
                  for t in PROTOCOLS for c in CELLS for m in MODELS for s in SEEDS], ignore_index=True)
save('phase12_training_logs.csv', logs)
ext_out = pd.DataFrame({'record_id': EXT.record_id, 'origin': EXT.origin})
for (tag, cell, arm), v in scores.items():
    if arm.startswith(('legnet', 'mprann')): ext_out[f'{tag}_{cell}_{arm}'] = v
save('phase12_external_scores_new_arms.csv', ext_out)
ckpts = {p.name: p for p in sorted(WEIGHTS.glob('*_best.pt'))}
assert len(ckpts) == len(CHOICES)
report = {'protocol_sha256': LOCK, 'config': CONFIG, 'smoke_test': SMOKE, 'device': str(DEVICE),
          'torch': torch.__version__, 'python': sys.version.split()[0],
          'result_sha256': {n: fsha(p) for n, p in files.items()},
          'checkpoints': {n: {'sha256': fsha(p), 'bytes': p.stat().st_size} for n, p in ckpts.items()},
          'limitations': ['200 bp genomic inputs without assay adapters; LegNet shift augmentation and reverse channel not used.',
                          'MPRAnn ported from Keras to PyTorch; eval-mode outputs matched the original with copied weights, but training dynamics (initialisation draws, optimiser numerics) are not bit-identical.',
                          'Hyperparameters are the published repository defaults, not tuned on these data.',
                          'Bootstrap intervals are conditional on the fitted models and frozen partitions; they do not capture all training variability. Seed-level results are reported separately.',
                          'External primary intervals resample sequences; a region-grouped bootstrap is reported as a sensitivity analysis.']}
(ART / 'phase12_provenance.json').write_text(json.dumps(report, indent=2, sort_keys=True) + '\n'); files['phase12_provenance.json'] = ART / 'phase12_provenance.json'
tagname = 'SMOKE_TEST_DO_NOT_REPORT_' if SMOKE else ''
OUT = WORK_BASE / f'{tagname}enhancer_phase12_strong_baselines_results.zip'
FULL = WORK_BASE / f'{tagname}enhancer_phase12_complete_with_weights.zip'
with zipfile.ZipFile(OUT, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as z:
    for n, p in files.items(): z.write(p, n)
with zipfile.ZipFile(FULL, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as z:
    z.write(OUT, OUT.name)
    for n, p in ckpts.items(): z.write(p, n)
for zp in (OUT, FULL):
    with zipfile.ZipFile(zp) as z: assert z.testzip() is None
print('READY:', OUT, fsha(OUT)); print('READY:', FULL, round(FULL.stat().st_size / 1e6, 1), 'MB')

READY: D:\Aditta Thesis\For Shafin\Shafin's Contribution\phase12_outputs\enhancer_phase12_strong_baselines_results.zip 7890a241aaa9d7b97e7bd321b491c16608ef086cd7035b05c69c4ff0a88dbb48
READY: D:\Aditta Thesis\For Shafin\Shafin's Contribution\phase12_outputs\enhancer_phase12_complete_with_weights.zip 254.3 MB
